# Recession Prediction Model Using Machine Learning

**Author:** [Your Name]  
**Date:** December 2024  
**Objective:** Build a machine learning model to predict US recession periods using financial market indicators

## Project Overview

This project develops a binary classification model to identify recession periods using:
- **Data Source:** Yahoo Finance, FRED (Federal Reserve Economic Data)
- **Time Period:** 2007-2024 (capturing 2008 financial crisis and 2020 COVID recession)
- **Target:** NBER official recession dates
- **Best Model:** XGBoost (ROC-AUC = 0.740)

### Key Features
- S&P 500 index
- VIX (volatility index)
- Treasury yield curve (2Y-10Y spread)
- Credit spreads (corporate bond risk premium)
- High-yield bond ETF (credit stress indicator)

### Skills Demonstrated
- Time series data handling
- Feature engineering (lags, rolling statistics, returns)
- Imbalanced classification
- Model comparison and evaluation
- Financial domain knowledge

---
## Step 1: Import Libraries and Setup

In [ ]:
# Core data manipulation
import pandas as pd
import numpy as np

# Data collection
import yfinance as yf
from pandas_datareader import data as pdr

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    precision_recall_curve,
    auc,
    roc_curve
)
import xgboost as xgb

# Utilities
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)

---
## Step 2: Data Collection

Collect financial market data from multiple sources:
- **Yahoo Finance:** Equity indices, VIX, ETFs
- **FRED:** Treasury yields, credit spreads

In [ ]:
print("=" * 60)
print("DATA COLLECTION")
print("=" * 60)

# Define date range (captures 2008 crisis and 2020 COVID recession)
START_DATE = "2007-01-01"
END_DATE = "2024-12-31"

print(f"\nDate Range: {START_DATE} to {END_DATE}")

### 2.1 Equity Market Data (Yahoo Finance)

In [ ]:
print("\nLoading equity market data...")

# Define tickers with descriptive names
tickers = {
    '^GSPC': 'sp500',              # S&P 500 Index
    '^VIX': 'vix',                 # Volatility Index (fear gauge)
    'HYG': 'high_yield_bonds',     # High-yield bond ETF (credit stress)
    'XLF': 'financials_sector',    # Financial sector ETF
    'GLD': 'gold',                 # Gold ETF (safe haven)
}

# Download all tickers
raw_data = yf.download(
    list(tickers.keys()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

# Extract close prices and rename columns
close_prices = raw_data['Close']
close_prices.rename(columns=tickers, inplace=True)

df_equity = close_prices.copy()

print(f"Equity data loaded: {df_equity.shape[0]} rows")
print(f"Features: {list(df_equity.columns)}")
print(f"\nFirst few rows:")
print(df_equity.head())

### 2.2 Treasury Yield Data (FRED)

In [ ]:
print("\nLoading Treasury yield data from FRED...")

fred_tickers = {
    'DGS10': 'treasury_10y',    # 10-Year Treasury Yield
    'DGS2': 'treasury_2y',      # 2-Year Treasury Yield
    'DGS3MO': 'treasury_3m',    # 3-Month Treasury Yield
}

bond_data = {}
for ticker, name in fred_tickers.items():
    try:
        df = pdr.DataReader(ticker, 'fred', start=START_DATE, end=END_DATE)
        bond_data[name] = df[ticker]
        print(f"   Downloaded {name.upper()}")
    except Exception as e:
        print(f"   Warning: Could not load {name}: {e}")

df_bonds = pd.DataFrame(bond_data)
print(f"\nBond data loaded: {df_bonds.shape[0]} rows")
print(df_bonds.head())

### 2.3 Credit Spread Data (FRED)

In [ ]:
print("\nLoading credit spread data from FRED...")

try:
    # BAA Corporate Bond Yield minus 10-Year Treasury
    credit_spread = pdr.DataReader('BAA10Y', 'fred', start=START_DATE, end=END_DATE)
    df_credit = credit_spread.rename(columns={'BAA10Y': 'credit_spread'})
    print(f"Credit spread loaded: {df_credit.shape[0]} rows")
except Exception as e:
    print(f"Warning: Could not load credit spread: {e}")
    df_credit = pd.DataFrame()

### 2.4 Merge All Datasets

In [ ]:
print("\nMerging all datasets...")

# Start with equity data
df = df_equity.copy()

# Merge bonds
df = df.join(df_bonds, how='outer')

# Merge credit spread
if not df_credit.empty:
    df = df.join(df_credit, how='outer')

print(f"Combined dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData summary:")
print(df.describe())

---
## Step 3: Data Cleaning

Handle missing values appropriately:
- Remove market holidays (completely empty rows)
- Forward-fill treasury data (holidays when bond market closed)
- Backward-fill remaining NaNs at start of dataset

In [ ]:
print("=" * 60)
print("DATA CLEANING")
print("=" * 60)

print(f"\nInitial missing data:")
print(df.isna().sum())

# Remove completely empty rows (market holidays)
completely_empty = df.isna().all(axis=1)
print(f"\nRemoving {completely_empty.sum()} market holiday rows...")
df = df[~completely_empty]

# Remove rows where all equity data is missing
equity_cols = ['gold', 'financials_sector', 'sp500', 'vix']
all_equity_missing = df[equity_cols].isna().all(axis=1)
print(f"Removing {all_equity_missing.sum()} rows with all equity data missing...")
df_clean = df[~all_equity_missing].copy()

# Forward fill remaining NaNs (carry last known values)
print(f"\nBefore forward fill:")
print(df_clean.isna().sum())

df_clean = df_clean.fillna(method='ffill')

# Backward fill any remaining NaNs at start
if df_clean.isna().any().any():
    print("\nBackward fill for remaining start-of-dataset NaNs...")
    df_clean = df_clean.fillna(method='bfill')

print(f"\nFinal result:")
print(f"  Remaining NaNs: {df_clean.isna().sum().sum()}")
print(f"  Final shape: {df_clean.shape}")
print(f"  Date range: {df_clean.index.min()} to {df_clean.index.max()}")

---
## Step 4: Feature Engineering

Create temporal features to capture market dynamics:
- **Lagged features:** Delayed effects (21-day, 63-day lags)
- **Rolling statistics:** Regime shifts (rolling mean, std)
- **Returns:** Momentum indicators (21-day, 63-day returns)
- **Yield curve:** 2Y-10Y spread (inversion indicator)

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Work on a copy
df_fe = df_clean.copy(deep=True)

print("\nCreating temporal features...")

# Create yield curve (short-term minus long-term)
if 'yield_curve' not in df_fe.columns:
    df_fe['yield_curve'] = df_fe['treasury_2y'] - df_fe['treasury_10y']
    print("  Created yield_curve feature")

# Configuration for feature generation
LAG_FEATURES = {
    'vix': [21, 63],
    'yield_curve': [21],
}

ROLLING_FEATURES = {
    'vix': {
        'mean': [21],
        'std': [21]
    },
    'yield_curve': {
        'mean': [63]
    }
}

RETURN_FEATURES = {
    'sp500': [21, 63],
    'gold': [21, 63]
}

# Generate lagged features
for feature, lags in LAG_FEATURES.items():
    for lag in lags:
        df_fe[f"{feature}_lag_{lag}d"] = df_fe[feature].shift(lag)
        print(f"  Created {feature}_lag_{lag}d")

# Generate rolling statistics
for feature, stats in ROLLING_FEATURES.items():
    for stat, windows in stats.items():
        for window in windows:
            if stat == 'mean':
                df_fe[f"{feature}_rolling_{stat}_{window}d"] = (
                    df_fe[feature].rolling(window=window).mean()
                )
            elif stat == 'std':
                df_fe[f"{feature}_rolling_{stat}_{window}d"] = (
                    df_fe[feature].rolling(window=window).std()
                )
            print(f"  Created {feature}_rolling_{stat}_{window}d")

# Generate return features
for feature, windows in RETURN_FEATURES.items():
    for window in windows:
        df_fe[f"{feature}_return_{window}d"] = df_fe[feature].pct_change(window)
        print(f"  Created {feature}_return_{window}d")

# Drop rows with NaN from rolling/lag operations
initial_rows = df_fe.shape[0]
df_fe = df_fe.dropna()
final_rows = df_fe.shape[0]

print(f"\nDropped {initial_rows - final_rows} rows due to rolling/lag features")
print(f"Final dataset shape: {df_fe.shape}")

---
## Step 5: Create Target Variable

Label recession periods using NBER official dates:
- **2007-12-01 to 2009-06-30:** Great Financial Crisis
- **2020-02-01 to 2020-04-30:** COVID-19 Recession

In [ ]:
print("=" * 60)
print("TARGET VARIABLE CREATION")
print("=" * 60)

# Ensure datetime index
df_fe.index = pd.to_datetime(df_fe.index)
if df_fe.index.tz is not None:
    df_fe.index = df_fe.index.tz_localize(None)

print(f"\nDate range: {df_fe.index.min()} to {df_fe.index.max()}")

# Define NBER recession periods
ALL_RECESSIONS = [
    ('2001-03-01', '2001-11-30'),  # Dot-com crash
    ('2007-12-01', '2009-06-30'),  # Great Financial Crisis
    ('2020-02-01', '2020-04-30'),  # COVID-19
]

# Filter to only recessions within our data range
data_start = df_fe.index.min()
data_end = df_fe.index.max()

NBER_RECESSIONS = []
for start, end in ALL_RECESSIONS:
    start_date = pd.to_datetime(start)
    end_date = pd.to_datetime(end)
    
    if start_date >= data_start and end_date <= data_end:
        NBER_RECESSIONS.append((start, end))
        print(f"  Including: {start} to {end}")
    else:
        print(f"  Skipping: {start} to {end} (outside data range)")

# Create binary target
df_fe['recession'] = 0

for start_date, end_date in NBER_RECESSIONS:
    mask = (df_fe.index >= start_date) & (df_fe.index <= end_date)
    df_fe.loc[mask, 'recession'] = 1
    days_marked = mask.sum()
    print(f"\nMarked {days_marked} days as recession: {start_date} to {end_date}")

# Analyze class distribution
recession_days = df_fe['recession'].sum()
total_days = len(df_fe)
recession_pct = (recession_days / total_days) * 100

print(f"\nTarget Variable Summary:")
print(f"  Total trading days: {total_days}")
print(f"  Recession days: {recession_days} ({recession_pct:.1f}%)")
print(f"  Normal days: {total_days - recession_days} ({100-recession_pct:.1f}%)")
print(f"  Class imbalance ratio: {(total_days - recession_days) / recession_days:.1f}:1")

if recession_pct < 15:
    print("\nNote: Highly imbalanced dataset!")
    print("  When modeling, must use:")
    print("    - class_weight='balanced' (Logistic Regression, Random Forest)")
    print("    - scale_pos_weight (XGBoost)")
    print("    - ROC-AUC and PR-AUC for evaluation (not accuracy)")

---
## Step 6: Exploratory Data Analysis

Visualize how features behave during recession vs normal periods

In [ ]:
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Create visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 14))
fig.suptitle('Feature Behavior During Recession Periods', 
             fontsize=16, fontweight='bold', y=0.995)

# Define recession periods for shading
recession_periods = [
    (pd.to_datetime('2007-12-01'), pd.to_datetime('2009-06-30')),
    (pd.to_datetime('2020-02-01'), pd.to_datetime('2020-04-30')),
]

# Helper function to add recession shading
def add_recession_shading(ax):
    for start, end in recession_periods:
        ax.axvspan(start, end, alpha=0.2, color='red', label='Recession')
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left')

# Plot 1: S&P 500
ax1 = axes[0]
ax1.plot(df_fe.index, df_fe['sp500'], linewidth=1.5, color='#2c3e50', label='S&P 500')
add_recession_shading(ax1)
ax1.set_ylabel('S&P 500 Index', fontsize=11, fontweight='bold')
ax1.set_title('1. Market Performance During Recessions', fontsize=12, pad=10)
ax1.grid(True, alpha=0.3)

# Plot 2: VIX
ax2 = axes[1]
ax2.plot(df_fe.index, df_fe['vix'], linewidth=1.5, color='#e74c3c', label='VIX (Volatility)')
ax2.axhline(y=20, color='orange', linestyle='--', alpha=0.5, label='Elevated Fear (20)')
ax2.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='High Fear (30)')
add_recession_shading(ax2)
ax2.set_ylabel('VIX Level', fontsize=11, fontweight='bold')
ax2.set_title('2. Market Fear (VIX) Spikes During Recessions', fontsize=12, pad=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Yield Curve
ax3 = axes[2]
ax3.plot(df_fe.index, df_fe['yield_curve'], linewidth=1.5, color='#3498db', 
         label='Yield Curve (2Y - 10Y)')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=2, alpha=0.7, label='Inversion Line')
ax3.fill_between(df_fe.index, 0, df_fe['yield_curve'], 
                 where=(df_fe['yield_curve'] < 0), 
                 color='orange', alpha=0.4, label='Inverted (Recession Warning)')
add_recession_shading(ax3)
ax3.set_ylabel('Yield Spread (%)', fontsize=11, fontweight='bold')
ax3.set_title('3. Yield Curve Inversion (Leading Indicator)', fontsize=12, pad=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Credit Spread
ax4 = axes[3]
ax4.plot(df_fe.index, df_fe['credit_spread'], linewidth=1.5, color='#9b59b6', 
         label='Credit Spread (BAA-10Y)')
ax4.axhline(y=df_fe['credit_spread'].median(), color='gray', linestyle='--', 
            alpha=0.5, label='Median Spread')
add_recession_shading(ax4)
ax4.set_ylabel('Credit Spread (%)', fontsize=11, fontweight='bold')
ax4.set_xlabel('Date', fontsize=11, fontweight='bold')
ax4.set_title('4. Credit Market Stress During Recessions', fontsize=12, pad=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('recession_feature_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'recession_feature_analysis.png'")

### Statistical Comparison: Recession vs Normal Periods

In [ ]:
comparison_features = ['vix', 'yield_curve', 'credit_spread', 'sp500_return_21d']

print("\nFeature Statistics: Recession vs Normal Periods\n")
print(f"{'Feature':<25} {'Normal Period':<20} {'Recession Period':<20} {'Difference':<15}")
print("-" * 80)

for feature in comparison_features:
    normal_mean = df_fe[df_fe['recession'] == 0][feature].mean()
    recession_mean = df_fe[df_fe['recession'] == 1][feature].mean()
    difference = recession_mean - normal_mean
    
    print(f"{feature:<25} {normal_mean:>18.2f} {recession_mean:>18.2f} {difference:>13.2f}")

print("\nKey Observations:")
print("  VIX is typically HIGHER during recessions (more fear)")
print("  Yield curve is more NEGATIVE before/during recessions (inversion)")
print("  Credit spreads WIDEN during recessions (more risk)")
print("  Stock returns are typically NEGATIVE during recessions")

---
## Step 7: Model Training Pipeline

Build a reusable pipeline to train and evaluate multiple models

In [ ]:
class RecessionModelPipeline:
    """
    Unified pipeline to train, evaluate, and compare multiple classification models.
    Handles class imbalance and provides comprehensive performance metrics.
    """
    
    def __init__(self, X_train, X_test, y_train, y_test, feature_names=None):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.feature_names = feature_names
        
        self.models = {}
        self.results = {}
        
        # Calculate scale_pos_weight for XGBoost
        self.scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        
        print(f"\nPipeline initialized:")
        print(f"  Training samples: {len(X_train)}")
        print(f"  Testing samples: {len(X_test)}")
        print(f"  Features: {X_train.shape[1]}")
        print(f"  Class imbalance ratio: {self.scale_pos_weight:.1f}:1")
    
    def train_logistic_regression(self):
        print("\nTraining Logistic Regression...")
        model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        model.fit(self.X_train, self.y_train)
        self.models['Logistic Regression'] = model
        self._evaluate_model('Logistic Regression', model)
        print("  Complete")
    
    def train_random_forest(self):
        print("\nTraining Random Forest...")
        model = RandomForestClassifier(
            n_estimators=100,
            class_weight='balanced',
            max_depth=10,
            min_samples_split=20,
            random_state=42,
            n_jobs=-1
        )
        model.fit(self.X_train, self.y_train)
        self.models['Random Forest'] = model
        self._evaluate_model('Random Forest', model)
        print("  Complete")
    
    def train_xgboost(self):
        print("\nTraining XGBoost...")
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=self.scale_pos_weight,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False
        )
        model.fit(self.X_train, self.y_train)
        self.models['XGBoost'] = model
        self._evaluate_model('XGBoost', model)
        print("  Complete")
    
    def _evaluate_model(self, model_name, model):
        y_pred = model.predict(self.X_test)
        y_pred_proba = model.predict_proba(self.X_test)[:, 1]
        
        roc_auc = roc_auc_score(self.y_test, y_pred_proba)
        precision, recall, _ = precision_recall_curve(self.y_test, y_pred_proba)
        pr_auc = auc(recall, precision)
        
        cm = confusion_matrix(self.y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        
        self.results[model_name] = {
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba,
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'confusion_matrix': cm,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
        }
    
    def train_all_models(self):
        print("\n" + "=" * 60)
        print("TRAINING ALL MODELS")
        print("=" * 60)
        self.train_logistic_regression()
        self.train_random_forest()
        self.train_xgboost()
        print("\nAll models trained successfully")
    
    def print_comparison_table(self):
        print("\n" + "=" * 60)
        print("MODEL PERFORMANCE COMPARISON")
        print("=" * 60)
        
        print(f"\n{'Model':<20} {'ROC-AUC':<12} {'PR-AUC':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print("-" * 80)
        
        for model_name, results in self.results.items():
            tp = results['tp']
            fp = results['fp']
            fn = results['fn']
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"{model_name:<20} {results['roc_auc']:<12.3f} {results['pr_auc']:<12.3f} "
                  f"{precision:<12.3f} {recall:<12.3f} {f1:<12.3f}")
        
        print("\nKey Metrics:")
        print("  ROC-AUC: Overall classification ability (>0.7 is good)")
        print("  PR-AUC: Performance on imbalanced data")
        print("  Precision: Of predicted recessions, how many were correct?")
        print("  Recall: Of actual recessions, how many did we catch?")
        print("  F1-Score: Harmonic mean of precision and recall")
    
    def plot_confusion_matrices(self):
        n_models = len(self.models)
        fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
        
        if n_models == 1:
            axes = [axes]
        
        for idx, (model_name, results) in enumerate(self.results.items()):
            cm = results['confusion_matrix']
            ax = axes[idx]
            im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
            ax.figure.colorbar(im, ax=ax)
            
            ax.set(xticks=[0, 1], yticks=[0, 1],
                   xticklabels=['Normal', 'Recession'],
                   yticklabels=['Normal', 'Recession'],
                   xlabel='Predicted',
                   ylabel='Actual',
                   title=f'{model_name}\nConfusion Matrix')
            
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, cm[i, j], ha="center", va="center", 
                           color="black", fontsize=14)
        
        plt.tight_layout()
        plt.savefig('model_confusion_matrices.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("\nConfusion matrices saved")
    
    def plot_roc_curves(self):
        plt.figure(figsize=(10, 8))
        
        for model_name, results in self.results.items():
            fpr, tpr, _ = roc_curve(self.y_test, results['y_pred_proba'])
            plt.plot(fpr, tpr, linewidth=2, 
                    label=f"{model_name} (AUC = {results['roc_auc']:.3f})")
        
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
        plt.legend(loc='lower right', fontsize=11)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('model_roc_curves.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("\nROC curves saved")
    
    def get_feature_importance(self, model_name='Random Forest', top_n=10):
        if model_name not in self.models:
            print(f"Error: Model '{model_name}' not found")
            return
        
        model = self.models[model_name]
        
        if not hasattr(model, 'feature_importances_'):
            print(f"Error: {model_name} does not support feature importance")
            return
        
        importances = model.feature_importances_
        
        if self.feature_names is not None:
            feature_imp = pd.DataFrame({
                'feature': self.feature_names,
                'importance': importances
            })
        else:
            feature_imp = pd.DataFrame({
                'feature': [f'Feature_{i}' for i in range(len(importances))],
                'importance': importances
            })
        
        feature_imp = feature_imp.sort_values('importance', ascending=False).head(top_n)
        
        plt.figure(figsize=(10, 6))
        plt.barh(range(len(feature_imp)), feature_imp['importance'], color='steelblue')
        plt.yticks(range(len(feature_imp)), feature_imp['feature'])
        plt.xlabel('Importance', fontsize=12)
        plt.title(f'Top {top_n} Feature Importances - {model_name}', 
                 fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'feature_importance_{model_name.lower().replace(" ", "_")}.png', 
                   dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"\nTop {top_n} features for {model_name}:")
        print(feature_imp.to_string(index=False))
        print("\nFeature importance plot saved")

print("\nModel pipeline class defined successfully")

### Prepare Data for Modeling

In [ ]:
print("=" * 60)
print("DATA PREPARATION FOR MODELING")
print("=" * 60)

# Separate features and target
X = df_fe.drop('recession', axis=1)
y = df_fe['recession']

feature_names = X.columns.tolist()

print(f"\nDataset summary:")
print(f"  Total samples: {len(X)}")
print(f"  Features: {X.shape[1]}")
print(f"  Recession samples: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"  Normal samples: {(1-y).sum()} ({(1-y.mean())*100:.1f}%)")

# Time-based train-test split
# Training: 2007-2019 (includes 2008-2009 recession)
# Testing: 2020-2024 (includes 2020 COVID recession)
train_end_date = '2019-12-31'

train_mask = df_fe.index <= train_end_date
test_mask = df_fe.index > train_end_date

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f"\nTrain-test split (time-based):")
print(f"  Training period: {X_train.index.min()} to {X_train.index.max()}")
print(f"  Testing period: {X_test.index.min()} to {X_test.index.max()}")
print(f"  Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Testing samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nClass distribution:")
print(f"  Train - Recession: {y_train.sum()} ({y_train.mean()*100:.1f}%), "
      f"Normal: {(1-y_train).sum()}")
print(f"  Test - Recession: {y_test.sum()} ({y_test.mean()*100:.1f}%), "
      f"Normal: {(1-y_test).sum()}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeatures scaled using StandardScaler")

### Train Models

In [ ]:
# Initialize pipeline
pipeline = RecessionModelPipeline(
    X_train=X_train_scaled,
    X_test=X_test_scaled,
    y_train=y_train,
    y_test=y_test,
    feature_names=feature_names
)

# Train all models
pipeline.train_all_models()

# Display results
pipeline.print_comparison_table()
pipeline.plot_confusion_matrices()
pipeline.plot_roc_curves()
pipeline.get_feature_importance(model_name='Random Forest', top_n=15)
pipeline.get_feature_importance(model_name='XGBoost', top_n=15)

---
## Step 8: Model Analysis Without Gold

Gold showed unexpected behavior (lower during 2020 recession vs 2008).
Retrain models without gold-related features to test impact.

In [ ]:
print("=" * 60)
print("RETRAIN WITHOUT GOLD FEATURES")
print("=" * 60)

# Remove gold-related features
gold_features = [col for col in X.columns if 'gold' in col.lower()]
print(f"\nRemoving features: {gold_features}")

X_no_gold = X.drop(columns=gold_features)

print(f"Original features: {X.shape[1]}")
print(f"After removing gold: {X_no_gold.shape[1]}")

# Same train-test split
X_train_ng = X_no_gold[train_mask]
X_test_ng = X_no_gold[test_mask]

# Scale features
scaler_ng = StandardScaler()
X_train_ng_scaled = scaler_ng.fit_transform(X_train_ng)
X_test_ng_scaled = scaler_ng.transform(X_test_ng)

# Initialize new pipeline
pipeline_no_gold = RecessionModelPipeline(
    X_train=X_train_ng_scaled,
    X_test=X_test_ng_scaled,
    y_train=y_train,
    y_test=y_test,
    feature_names=X_no_gold.columns.tolist()
)

# Train models
pipeline_no_gold.train_random_forest()
pipeline_no_gold.train_xgboost()

# Compare results
pipeline_no_gold.print_comparison_table()
pipeline_no_gold.plot_confusion_matrices()
pipeline_no_gold.plot_roc_curves()
pipeline_no_gold.get_feature_importance(model_name='Random Forest', top_n=15)
pipeline_no_gold.get_feature_importance(model_name='XGBoost', top_n=15)

# Comparison table
print("\n" + "=" * 60)
print("COMPARISON: WITH vs WITHOUT GOLD")
print("=" * 60)

print(f"\n{'Model':<25} {'With Gold ROC-AUC':<20} {'Without Gold ROC-AUC':<20}")
print("-" * 65)

for model_name in ['Random Forest', 'XGBoost']:
    with_gold_auc = pipeline.results[model_name]['roc_auc']
    without_gold_auc = pipeline_no_gold.results[model_name]['roc_auc']
    print(f"{model_name:<25} {with_gold_auc:<20.3f} {without_gold_auc:<20.3f}")

---
## Step 9: Final Summary and Conclusions

In [ ]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

final_summary = """
RECESSION PREDICTION MODEL - PROJECT SUMMARY

OBJECTIVE:
Build a machine learning model to predict US recession periods using financial indicators

DATA:
- Period: 2007-2024 (18 years)
- Samples: 2,868 (after feature engineering)
- Training: 2007-2019 (includes 2008-2009 recession)
- Testing: 2020-2024 (includes 2020 COVID recession)
- Class distribution: 11.1% recession, 88.9% normal

FEATURES:
Base indicators:
- S&P 500, VIX, yield curve, credit spreads, treasuries, high-yield bonds

Engineered features:
- Temporal lags (21-day, 63-day)
- Rolling statistics (mean, std)
- Momentum indicators (returns)
- Total: 20 features (17 without gold)

MODEL PERFORMANCE:

Best Model: XGBoost (without gold)
- ROC-AUC: 0.740
- Top features: high_yield_bonds (82%), credit_spread (10%), yield_curve (7%)

Random Forest (without gold)
- ROC-AUC: 0.562
- Top features: high_yield_bonds (30%), sp500 (12%), vix_rolling_mean (11%)

KEY INSIGHTS:
1. Credit market stress (high-yield bonds) is the strongest recession predictor
2. Gold was misleading - different behavior in 2008 vs 2020
3. Yield curve and credit spreads provide valuable early signals
4. VIX momentum captures fear buildup better than raw VIX

LIMITATIONS:
1. Only one recession event in test set (2020 COVID)
2. Model trained primarily on 2008 financial crisis patterns
3. Limited sample size of recession periods overall
4. 2020 COVID recession was unique (policy-driven, shortest ever)

IMPROVEMENTS:
1. Cross-validation with different time periods
2. Additional features (market breadth, sector rotation)
3. Ensemble of multiple models
4. Threshold optimization for precision/recall balance
5. Real-time testing on future data

CONCLUSION:
Successfully built a recession prediction model with ROC-AUC of 0.74.
Model identifies credit market stress as the primary leading indicator.
Demonstrates proficiency in time-series ML, feature engineering, and financial domain knowledge.
"""

print(final_summary)

# Save summary
with open('project_summary.txt', 'w') as f:
    f.write(final_summary)

print("\nProject summary saved to 'project_summary.txt'")

### Create Final Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Model comparison
models = ['Random Forest\n(with gold)', 'XGBoost\n(with gold)', 
          'Random Forest\n(no gold)', 'XGBoost\n(no gold)']
roc_scores = [0.722, 0.733, 0.562, 0.740]
colors = ['lightcoral', 'lightcoral', 'steelblue', 'steelblue']

ax1 = axes[0]
bars = ax1.bar(models, roc_scores, color=colors, alpha=0.7, edgecolor='black')
ax1.axhline(y=0.5, color='red', linestyle='--', label='Random baseline')
ax1.axhline(y=0.7, color='green', linestyle='--', label='Good performance', alpha=0.5)
ax1.set_ylabel('ROC-AUC Score')
ax1.set_title('Model Performance Comparison')
ax1.set_ylim([0, 1])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

for bar, score in zip(bars, roc_scores):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Feature importance
top_features = ['high_yield_bonds', 'credit_spread', 'yield_curve_rolling_mean_63d', 
                'treasury_3m', 'sp500']
importances = [0.818, 0.102, 0.072, 0.018, 0.015]

ax2 = axes[1]
ax2.barh(top_features, importances, color='steelblue', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Feature Importance')
ax2.set_title('Top 5 Features - XGBoost (Best Model)')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

for i, (feature, imp) in enumerate(zip(top_features, importances)):
    ax2.text(imp, i, f' {imp:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('final_model_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFinal visualization saved as 'final_model_summary.png'")

print("\n" + "=" * 60)
print("PROJECT COMPLETE")
print("=" * 60)
print("\nDeliverables:")
print("1. Cleaned dataset (2,868 samples)")
print("2. 20 engineered features")
print("3. Binary recession target (NBER dates)")
print("4. 3 trained models (Logistic Regression, Random Forest, XGBoost)")
print("5. Best model: XGBoost (ROC-AUC = 0.740)")
print("6. Comprehensive visualizations")
print("7. Feature importance analysis")
print("8. Model comparison")
print("\nREADY FOR PORTFOLIO")